<a href="https://colab.research.google.com/github/suphanatchanlek30/Super-AI-Engineer-Season-6-Mini-Hackathon-Week-4-5-Thai-Language-Image-Captioning/blob/main/Thai_Language_Image_Captioning_600367_%E0%B8%A8%E0%B8%B8%E0%B8%A0%E0%B8%93%E0%B8%B1%E0%B8%90_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Cell 1

In [ ]:
from google.colab import files
uploaded = files.upload()  # upload kaggle.json only

Saving kaggle.json to kaggle.json


Cell 2

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json
!ls -la ~/.kaggle


total 16
drwxr-xr-x 2 root root 4096 Apr  4 04:42 .
drwx------ 1 root root 4096 Apr  4 04:42 ..
-rw------- 1 root root   71 Apr  4 04:42 kaggle.json


Cell 3

In [ ]:
!pip -q install --no-cache-dir \
  kaggle==1.7.4.5 \
  sacrebleu==2.5.1 \
  sentencepiece==0.2.0 \
  transformers==4.53.3 \
  accelerate==1.8.1


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 18.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 141.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.2/181.2 kB 90.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 86.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 162.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.3/365.3 kB 188.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 161.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 110.1 MB/s eta 0:00:00


Cell 4

In [ ]:
!mkdir -p /content/competition_data
!kaggle competitions download -c super-ai-engineer-ss-6-thai-language-image-captioning -p /content/competition_data
!unzip -qo /content/competition_data/super-ai-engineer-ss-6-thai-language-image-captioning.zip -d /content/competition_data
!find /content/competition_data -maxdepth 3 | head -100


 98% 1.71G/1.75G [00:21<00:00, 233MB/s]
100% 1.75G/1.75G [00:21<00:00, 85.5MB/s]
/content/competition_data
/content/competition_data/super-ai-engineer-ss-6-thai-language-image-captioning.zip
/content/competition_data/capgen_v1.0_train.json
/content/competition_data/val
/content/competition_data/val/val
/content/competition_data/val/val/travel
/content/competition_data/val/val/food
/content/competition_data/train
/content/competition_data/train/train
/content/competition_data/train/train/travel
/content/competition_data/train/train/food
/content/competition_data/sample_submission.csv
/content/competition_data/capgen_v1.0_val.json
/content/competition_data/test
/content/competition_data/test/test
/content/competition_data/test/test/01045.jpg
/content/competition_data/test/test/00216.jpg
/content/competition_data/test/test/01593.jpg
/content/competition_data/test/test/01924.jpg
/content/competition_data/test/test/00048.jpg
/content/competition_data/test/test/00951.jpg
/content/competition

Cell 5

In [ ]:
import gc
import json
import random
import re
import sys
import unicodedata
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import PIL
import torch
from PIL import Image
from sacrebleu.metrics import BLEU
from tqdm.auto import tqdm
from transformers import AutoProcessor, CLIPModel

print("python =", sys.version)
print("numpy =", np.__version__)
print("pandas =", pd.__version__)
print("pillow =", PIL.__version__)
print("torch =", torch.__version__)
print("cuda =", torch.cuda.is_available())

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE =", DEVICE)

@dataclass
class CFG:
    data_root: str = "/content/competition_data"
    use_coco_in_retrieval: bool = False
    retrieval_topk_grid: Tuple[int, ...] = (1, 3, 5, 8, 12)
    image_batch_size: int = 64
    model_name: str = "openai/clip-vit-base-patch32"
    source_weight_ipu24: float = 1.10
    source_weight_coco: float = 0.92
    similarity_weight: float = 0.78
    consensus_weight: float = 0.22
    max_candidates_per_query: int = 48
    output_submission: str = "/content/submission.csv"

cfg = CFG()
DATA_ROOT = Path(cfg.data_root)


python = 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
numpy = 2.0.2
pandas = 2.2.2
pillow = 11.3.0
torch = 2.10.0+cpu
cuda = False
DEVICE = cpu


Cell 6

In [ ]:
TRAIN_JSON = DATA_ROOT / "capgen_v1.0_train.json"
VAL_JSON = DATA_ROOT / "capgen_v1.0_val.json"
SAMPLE_SUBMISSION = DATA_ROOT / "sample_submission.csv"

print(TRAIN_JSON.exists(), TRAIN_JSON)
print(VAL_JSON.exists(), VAL_JSON)
print(SAMPLE_SUBMISSION.exists(), SAMPLE_SUBMISSION)

if not TRAIN_JSON.exists():
    raise FileNotFoundError("capgen_v1.0_train.json not found")
if not VAL_JSON.exists():
    raise FileNotFoundError("capgen_v1.0_val.json not found")
if not SAMPLE_SUBMISSION.exists():
    raise FileNotFoundError("sample_submission.csv not found")

with open(TRAIN_JSON, "r", encoding="utf-8") as f:
    train_map = json.load(f)

with open(VAL_JSON, "r", encoding="utf-8") as f:
    val_map = json.load(f)

sample_submission = pd.read_csv(SAMPLE_SUBMISSION)

print("train images =", len(train_map))
print("val images =", len(val_map))
print("test rows =", len(sample_submission))
sample_submission.head()


True /content/competition_data/capgen_v1.0_train.json
True /content/competition_data/capgen_v1.0_val.json
True /content/competition_data/sample_submission.csv
train images = 142291
val images = 9036
test rows = 2000


,image_id,caption
0,1354,ภาพถ่ายระยะใกล้ของวัตถุทรงกลมสีขาวที่มีคราบสีด...
1,1413,นกตัวหนึ่งที่กําลังเกาะอยู่บนกิ่งไม้อันหนึ่งที...
2,1802,บ้านที่อยู่ติดกับบริเวณริมชายหาดทะเลและมีต้นไม...
3,1243,NaN
4,693,NaN


Cell 7

In [ ]:
def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFC", str(text))
    text = text.replace("\u200b", " ").replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

def ensure_3_refs(refs: List[str]) -> List[str]:
    refs = [normalize_text(x) for x in refs if normalize_text(x)]
    if not refs:
        refs = [""]
    while len(refs) < 3:
        refs.append(refs[-1])
    return refs[:3]

def path_source(rel_path: str) -> str:
    if rel_path.startswith("ipu24/"):
        return "ipu24"
    if rel_path.startswith("coco/"):
        return "coco"
    return "unknown"

def path_domain(rel_path: str) -> str:
    parts = rel_path.split("/")
    if len(parts) >= 3 and parts[0] == "ipu24":
        return parts[2]
    if len(parts) >= 2 and parts[0] == "coco":
        return parts[1]
    return "unknown"

def resolve_image_path(data_root: Path, rel_path: str, split_name: str) -> Path:
    rel = Path(rel_path)

    candidates = [
        data_root / rel_path,
        data_root / split_name / rel_path,
        data_root / split_name / split_name / rel_path,
        data_root / "train" / rel_path,
        data_root / "train" / "train" / rel_path,
        data_root / "val" / rel_path,
        data_root / "val" / "val" / rel_path,
        data_root / "test" / rel_path,
        data_root / "test" / "test" / rel_path,
    ]

    for c in candidates:
        if c.exists():
            return c

    name = rel.name
    matches = list(data_root.rglob(name))

    if len(matches) == 1:
        return matches[0]

    if len(matches) > 1:
        rel_norm = rel_path.replace("\\", "/")
        split_token = f"/{split_name}/"

        scored = []
        for m in matches:
            m_str = str(m).replace("\\", "/")
            score = 0
            if rel_norm in m_str:
                score += 10
            if split_token in m_str:
                score += 3
            if path_source(rel_path) in m_str:
                score += 2
            scored.append((score, m))

        scored.sort(key=lambda x: x[0], reverse=True)
        best = scored[0][1]
        return best

    raise FileNotFoundError(f"Cannot resolve image path: {rel_path}")

def build_image_records(annotation_map: Dict[str, List[str]], split_name: str, data_root: Path) -> pd.DataFrame:
    rows = []
    missing = []

    for rel_path, captions in tqdm(annotation_map.items(), desc=f"building {split_name}"):
        try:
            image_path = resolve_image_path(data_root, rel_path, split_name)
            rows.append({
                "split": split_name,
                "rel_path": rel_path,
                "image_path": str(image_path),
                "source": path_source(rel_path),
                "domain": path_domain(rel_path),
                "captions": ensure_3_refs(captions),
            })
        except FileNotFoundError:
            missing.append(rel_path)

    df = pd.DataFrame(rows)

    print(f"{split_name}: resolved {len(df)} images")
    print(f"{split_name}: missing  {len(missing)} images")
    if missing:
        print("sample missing:", missing[:10])

    return df


Cell 8

In [ ]:
train_df = build_image_records(train_map, "train", DATA_ROOT)
val_df = build_image_records(val_map, "val", DATA_ROOT)

print(train_df.head(3))
print(train_df["source"].value_counts(dropna=False))
print(train_df["domain"].value_counts(dropna=False).head(20))


building train:   0%|          | 0/142291 [00:00<?, ?it/s]

Cell 9

In [ ]:
bleu_metric = BLEU(tokenize="flores101", effective_order=True)

def corpus_sacrebleu(preds: List[str], refs_list: List[List[str]]) -> float:
    clean_preds = [normalize_text(x) for x in preds]
    clean_refs = [ensure_3_refs(x) for x in refs_list]
    ref_streams = [[x[i] for x in clean_refs] for i in range(3)]
    return bleu_metric.corpus_score(clean_preds, ref_streams).score

def sentence_sacrebleu(pred: str, refs: List[str]) -> float:
    refs = ensure_3_refs(refs)
    return bleu_metric.sentence_score(normalize_text(pred), refs).score

print(sentence_sacrebleu(train_df.iloc[0]["captions"][0], train_df.iloc[0]["captions"]))


Cell 10

In [ ]:
train_ipu24 = train_df[train_df["source"] == "ipu24"].reset_index(drop=True)
val_ipu24 = val_df[val_df["source"] == "ipu24"].reset_index(drop=True)

if cfg.use_coco_in_retrieval:
    retrieval_db_train = train_df.copy().reset_index(drop=True)
else:
    retrieval_db_train = train_ipu24.copy().reset_index(drop=True)

print("retrieval_db_train =", len(retrieval_db_train))
print("val_ipu24 =", len(val_ipu24))

processor = AutoProcessor.from_pretrained(cfg.model_name)
vision_model = CLIPModel.from_pretrained(cfg.model_name).to(DEVICE)
vision_model.eval()

print("Loaded:", cfg.model_name)


Cell 11

In [ ]:
def load_rgb(path: str) -> Image.Image:
    return Image.open(path).convert("RGB")

@torch.no_grad()
def encode_image_paths(image_paths: List[str], batch_size: int = 64) -> np.ndarray:
    all_features = []
    for start in tqdm(range(0, len(image_paths), batch_size), desc="Encoding images"):
        batch_paths = image_paths[start:start + batch_size]
        images = [load_rgb(p) for p in batch_paths]
        inputs = processor(images=images, return_tensors="pt")
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        feats = vision_model.get_image_features(**inputs)
        feats = feats / feats.norm(dim=-1, keepdim=True)
        all_features.append(feats.detach().cpu().numpy().astype(np.float32))

        del images, inputs, feats
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    return np.concatenate(all_features, axis=0)


Cell 12

In [ ]:
train_db_emb = encode_image_paths(
    retrieval_db_train["image_path"].tolist(),
    batch_size=cfg.image_batch_size
)

val_query_emb = encode_image_paths(
    val_ipu24["image_path"].tolist(),
    batch_size=cfg.image_batch_size
)

print("train_db_emb =", train_db_emb.shape)
print("val_query_emb =", val_query_emb.shape)


Cell 13

In [ ]:
def cosine_similarity_matrix(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    return np.matmul(a, b.T)

def build_caption_pool(row: pd.Series) -> List[str]:
    return [normalize_text(x) for x in row["captions"] if normalize_text(x)]

def neighbor_weight(source: str) -> float:
    if source == "ipu24":
        return cfg.source_weight_ipu24
    if source == "coco":
        return cfg.source_weight_coco
    return 1.0

def choose_caption_from_neighbors(sims, db_df, top_k, max_candidates):
    top_idx = np.argsort(-sims)[:top_k]

    candidate_items = []
    for rank, idx in enumerate(top_idx):
        row = db_df.iloc[idx]
        sim = max(float(sims[idx]), 0.0)
        w = sim * neighbor_weight(row["source"])

        for cap in build_caption_pool(row):
            candidate_items.append({
                "caption": cap,
                "img_score": w,
                "rank": rank,
            })

    if not candidate_items:
        return "ภาพหนึ่งภาพ"

    candidate_items = candidate_items[:max_candidates]
    unique_candidates = list(dict.fromkeys(x["caption"] for x in candidate_items))

    best_caption = unique_candidates[0]
    best_score = -1e18

    for cand in unique_candidates:
        sim_term = 0.0
        consensus_term = 0.0
        support = 0

        for item in candidate_items:
            if item["caption"] == cand:
                sim_term = max(sim_term, item["img_score"])
                continue

            support += 1
            pair_bleu = sentence_sacrebleu(cand, [item["caption"]])
            consensus_term += pair_bleu * item["img_score"]

        if support > 0:
            consensus_term /= support

        length_penalty = 0.0
        token_count = len(cand.split())
        if token_count < 4:
            length_penalty -= 1.5
        elif token_count > 22:
            length_penalty -= 1.5

        final_score = (
            cfg.similarity_weight * sim_term
            + cfg.consensus_weight * consensus_term
            + length_penalty
        )

        if final_score > best_score:
            best_score = final_score
            best_caption = cand

    return normalize_text(best_caption)

def predict_from_embeddings(query_emb, db_emb, db_df, top_k, max_candidates):
    sims = cosine_similarity_matrix(query_emb, db_emb)
    preds = []
    for i in tqdm(range(len(query_emb)), desc=f"Predicting top_k={top_k}"):
        preds.append(
            choose_caption_from_neighbors(
                sims=sims[i],
                db_df=db_df,
                top_k=top_k,
                max_candidates=max_candidates,
            )
        )
    return preds


Cell 14

In [ ]:
val_refs = val_ipu24["captions"].tolist()

grid_rows = []
best_topk = None
best_bleu = -1.0
best_val_preds = None

for top_k in cfg.retrieval_topk_grid:
    preds = predict_from_embeddings(
        query_emb=val_query_emb,
        db_emb=train_db_emb,
        db_df=retrieval_db_train,
        top_k=top_k,
        max_candidates=cfg.max_candidates_per_query,
    )
    score = corpus_sacrebleu(preds, val_refs)
    grid_rows.append({"top_k": top_k, "bleu": score})
    print(f"top_k={top_k:<2d}  val_sacrebleu={score:.5f}")

    if score > best_bleu:
        best_bleu = score
        best_topk = top_k
        best_val_preds = preds

grid_df = pd.DataFrame(grid_rows).sort_values("bleu", ascending=False).reset_index(drop=True)
grid_df


Cell 15

In [ ]:
print("BEST top_k =", best_topk)
print("BEST BLEU =", best_bleu)

preview_df = val_ipu24[["rel_path", "captions"]].copy()
preview_df["prediction"] = best_val_preds
preview_df.head(10)


Cell 16

In [ ]:
if cfg.use_coco_in_retrieval:
    final_db = pd.concat([train_df, val_df], ignore_index=True)
else:
    final_db = pd.concat([train_ipu24, val_ipu24], ignore_index=True)

print("final_db size =", len(final_db))
print(final_db["source"].value_counts(dropna=False))


Cell 17

In [ ]:
def resolve_test_path(image_id: str, data_root: Path) -> str:
    image_id = str(image_id).zfill(5)

    direct = data_root / "test" / "test" / f"{image_id}.jpg"
    if direct.exists():
        return str(direct)

    matches = list(data_root.rglob(f"{image_id}.jpg"))
    if not matches:
        raise FileNotFoundError(f"Test image not found for image_id={image_id}")
    return str(matches[0])

test_df = sample_submission.copy()
test_df["image_id"] = test_df["image_id"].astype(str).str.zfill(5)
test_df["image_path"] = test_df["image_id"].apply(lambda x: resolve_test_path(x, DATA_ROOT))
test_df.head()


Cell 18

In [ ]:
final_db_emb = encode_image_paths(
    final_db["image_path"].tolist(),
    batch_size=cfg.image_batch_size
)

test_emb = encode_image_paths(
    test_df["image_path"].tolist(),
    batch_size=cfg.image_batch_size
)

print("final_db_emb =", final_db_emb.shape)
print("test_emb =", test_emb.shape)


Cell 19

In [ ]:
test_preds = predict_from_embeddings(
    query_emb=test_emb,
    db_emb=final_db_emb,
    db_df=final_db,
    top_k=best_topk,
    max_candidates=cfg.max_candidates_per_query,
)

test_preds[:10]


Cell 20

In [ ]:
submission = sample_submission.copy()
submission["image_id"] = submission["image_id"].astype(str).str.zfill(5)
submission["caption"] = [normalize_text(x) for x in test_preds]
submission.to_csv(cfg.output_submission, index=False, encoding="utf-8")

print("Saved submission to", cfg.output_submission)
submission.head(10)


Cell 21


In [ ]:
from google.colab import files
files.download(cfg.output_submission)


Cell 22

In [ ]:
val_oof = val_ipu24.copy()
val_oof["prediction"] = best_val_preds
val_oof["best_topk"] = best_topk
val_oof.to_json("/content/val_predictions.json", orient="records", force_ascii=False, indent=2)

print("Saved /content/val_predictions.json")

Cell 23

In [ ]:
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()
print("Finished.")